In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_sample_weight

#loading data
train = pd.read_csv('/Users/amritdhillon/Desktop/Advanced ML/Kaggle Playground Series Competition/train.csv')
test = pd.read_csv('/Users/amritdhillon/Desktop/Advanced ML/Kaggle Playground Series Competition/test.csv')

#same feature engineering as XGBoost model I got good results on earlier
train['Moisture_to_Heat_Ratio'] = train['Soil_Moisture'] / (train['Temperature_C'] + 1)
test['Moisture_to_Heat_Ratio'] = test['Soil_Moisture'] / (test['Temperature_C'] + 1)

train['Water_Availability_Index'] = (train['Rainfall_mm'] + train['Previous_Irrigation_mm']) / (train['Field_Area_hectare'] + 1)
test['Water_Availability_Index'] = (test['Rainfall_mm'] + test['Previous_Irrigation_mm']) / (test['Field_Area_hectare'] + 1)

train['Climate_Stress_Index'] = (train['Temperature_C'] * train['Sunlight_Hours'] * (1 + train['Wind_Speed_kmh'] / 25)) / (train['Humidity'] + 1)
test['Climate_Stress_Index'] = (test['Temperature_C'] * test['Sunlight_Hours'] * (1 + test['Wind_Speed_kmh'] / 25)) / (test['Humidity'] + 1)

#splitting features and target
X = train.drop(columns=['Irrigation_Need', 'id'])
y = train['Irrigation_Need']
X_test = test.drop(columns=['id'])

#encode target
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

#encode categorical features since Random Forest needs numeric
X = pd.get_dummies(X)
X_test = pd.get_dummies(X_test)

#align columns of train and test because get_dummies can create different columns if there are categories in one set that aren't in the other
X, X_test = X.align(X_test, join='left', axis=1, fill_value=0)

#class weights
sample_weights = compute_sample_weight(class_weight='balanced', y=y_encoded)

#CV
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

#Random Forest model
rf = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

#grid search for hyperparameter tuning for RandomizedSearchCV
param_dist_rf = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 15, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt', 'log2']
}

random_rf = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist_rf,
    n_iter=8,
    scoring='balanced_accuracy',
    cv=skf,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

random_rf.fit(X, y_encoded, sample_weight=sample_weights)

print("Best RF params:", random_rf.best_params_)
print("Best RF CV score:", random_rf.best_score_)

Fitting 3 folds for each of 8 candidates, totalling 24 fits


Exception ignored in: <function ResourceTracker.__del__ at 0x10526dbc0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x102a9dbc0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x106aa5bc0>
Traceback (most recent call last

Best RF params: {'n_estimators': 200, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'max_depth': 10}
Best RF CV score: 0.9631437997628631


In [2]:
best_rf = random_rf.best_estimator_

best_rf.fit(X, y_encoded, sample_weight=sample_weights)

rf_preds_encoded = best_rf.predict(X_test)
rf_preds = label_encoder.inverse_transform(rf_preds_encoded)

submission_rf = pd.DataFrame({
    'id': test['id'],
    'Irrigation_Need': rf_preds
})

submission_rf.to_csv(
    '/Users/amritdhillon/Desktop/Advanced ML/Kaggle Playground Series Competition/model outputs/submission_rf.csv',
    index=False
)

submission_rf.head()

,id,Irrigation_Need
0,630000,Low
1,630001,Low
2,630002,Low
3,630003,Low
4,630004,Low


Exception ignored in: <function ResourceTracker.__del__ at 0x102449bc0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x110065bc0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes


In [4]:
#just checking for submission
row_count = len(submission_rf)
print(f'Total rows: {row_count}')

Total rows: 270000


Exception ignored in: <function ResourceTracker.__del__ at 0x102dadbc0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x103a09bc0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x105bddbc0>
Traceback (most recent call last